In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import string

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [24]:
path = "/content/restaurant_reviews_50000.csv"

In [25]:
df = pd.read_csv(path)

In [5]:
import re
def remove_html(text):
    return re.sub(r'<.*?>', '', text)

In [6]:
df['review'] = df['review'].apply(remove_html)

In [9]:
import re

def remove_urls(text):
    return re.sub(r'(https?://|www\.)\S+', '', text)

In [10]:
df['review'] = df['review'].apply(remove_urls)

In [29]:
df['positive_or_negative'].value_counts()

,count
positive_or_negative,
positive,25000
negative,25000


In [18]:
df.isnull().sum()

,0
review,0
sentiment,0


In [31]:
df.duplicated().sum()

np.int64(0)

In [30]:
df.drop_duplicates(inplace=True)

In [32]:
df.duplicated().sum()

np.int64(0)

In [36]:
df['review'] = df['review'].str.lower()

In [37]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
sw_list = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [38]:
df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

,review
0,great experience grilled chicken better expect...
1,impressed lemonade staff rude inattentive ingr...
2,great experience cheesecake texture excellent ...
3,tables clean music nice impressed chocolate ca...
4,good experience burger everyone welcoming help...
...,...
49995,absolutely loved pasta portion generous fresh ...
49996,great experience vegetable couscous arrived ho...
49997,recommend seafood platter much worse expected ...
49998,really unhappy vegetable couscous flavors unba...


In [39]:
df = df.iloc[:10000]

In [40]:
x = df['review']
y = df['positive_or_negative']

In [41]:
x.head()

,review
0,great experience grilled chicken because bette...
1,not impressed lemonade staff rude inattentive ...
2,great experience cheesecake texture excellent ...
3,tables clean music nice impressed chocolate ca...
4,such good experience burger everyone welcoming...


In [42]:
y

,positive_or_negative
0,positive
1,negative
2,positive
3,positive
4,positive
...,...
10004,negative
10005,negative
10006,negative
10007,positive


In [43]:
#Label encoding for y
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [45]:
y = encoder.fit_transform(y)

In [46]:
y

array([1, 0, 1, ..., 0, 1, 1])

In [35]:
df.head()

,review,positive_or_negative
0,great experience grilled chicken because bette...,positive
1,not impressed lemonade staff rude inattentive ...,negative
2,great experience cheesecake texture excellent ...,positive
3,tables clean music nice impressed chocolate ca...,positive
4,such good experience burger everyone welcoming...,positive


Split data into testing (20%) and training (80%)

In [47]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=1)

In [48]:
x_train.shape

(8000,)

In [49]:
x_test.shape

(2000,)

In [51]:
y_train.shape

(8000,)

In [52]:
y_test.shape

(2000,)

In [57]:
#Bag of word
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [58]:
x_train_bow = cv.fit_transform(x_train).toarray()
x_test_bow = cv.transform(x_test).toarray()

In [59]:
x_train_bow

array([[0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 1, 0]])

Naive Bayes

In [61]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

gnb.fit(x_train_bow,y_train)

GaussianNB()

In [63]:
y_pred = gnb.predict(x_test_bow)

from sklearn.metrics import accuracy_score,confusion_matrix
accuracy_score(y_test,y_pred)

1.0

In [64]:
confusion_matrix(y_test,y_pred)

array([[ 985,    0],
       [   0, 1015]])

Random Forest

In [65]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

rf.fit(x_train_bow,y_train)

y_pred = rf.predict(x_test_bow)

accuracy_score(y_test,y_pred)

1.0

In [67]:
cv = CountVectorizer(max_features=3000)

x_train_bow = cv.fit_transform(x_train).toarray()
x_test_bow = cv.transform(x_test).toarray()

rf = RandomForestClassifier()

rf.fit(x_train_bow,y_train)
y_pred = rf.predict(x_test_bow)
accuracy_score(y_test,y_pred)


1.0

N Gram

In [68]:
cv = CountVectorizer(ngram_range=(1,2),max_features=5000)

x_train_bow = cv.fit_transform(x_train).toarray()
x_test_bow = cv.transform(x_test).toarray()

rf = RandomForestClassifier()

rf.fit(x_train_bow,y_train)
y_pred = rf.predict(x_test_bow)
accuracy_score(y_test,y_pred)

1.0

TFIDF

In [70]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

x_train_tfidf = cv.fit_transform(x_train).toarray()
x_test_tfidf = cv.transform(x_test).toarray()


rf = RandomForestClassifier()

rf.fit(x_train_tfidf,y_train)
y_pred = rf.predict(x_test_tfidf)

accuracy_score(y_test,y_pred)

1.0